# ASR MODEL USING HMM

In [14]:
import os
import numpy as np
import librosa
from hmmlearn import hmm
import warnings
warnings.filterwarnings("ignore")

In [15]:
N_MFCC = 13
HMM_COMPONENTS = 5
DATA_DIR = "data" 

In [16]:
def extract_mfcc(file_path, n_mfcc=N_MFCC):
    y, sr = librosa.load(file_path, sr=None)
    y, _ = librosa.effects.trim(y)
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
    return mfcc.T

In [17]:
def train_models(data_dir):
    models = {}
    for word in os.listdir(data_dir):
        word_path = os.path.join(data_dir, word)
        if not os.path.isdir(word_path): continue

        print(f"Training HMM for word: '{word}'")
        all_feats = []
        lengths = []

        for fname in os.listdir(word_path):
            if not fname.endswith(".wav"): continue
            path = os.path.join(word_path, fname)
            feats = extract_mfcc(path)
            all_feats.append(feats)
            lengths.append(len(feats))

        X = np.concatenate(all_feats)
        model = hmm.GaussianHMM(n_components=HMM_COMPONENTS, covariance_type='diag', n_iter=100)
        model.fit(X, lengths)
        models[word] = model

    return models

In [18]:
def recognize(models, test_file):
    feats = extract_mfcc(test_file)
    scores = {}
    for word, model in models.items():
        try:
            scores[word] = model.score(feats)
        except:
            scores[word] = -np.inf

    best_word = max(scores, key=scores.get)
    return best_word, scores

In [19]:
if __name__ == "__main__":
    models = train_models(DATA_DIR)
    test_file = "test.wav"
    predicted_word, all_scores = recognize(models, test_file)

    print(f"\nPredicted word: {predicted_word}")
    print("Likelihood scores:")
    for word, score in all_scores.items():
        print(f"  {word}: {score:.2f}")

Training HMM for word: 'five'
Training HMM for word: 'four'
Training HMM for word: 'one'
Training HMM for word: 'six'
Training HMM for word: 'three'
Training HMM for word: 'two'
Training HMM for word: 'zero'

Predicted word: three
Likelihood scores:
  five: -684.66
  four: -820.23
  one: -612.14
  six: -596.12
  three: -449.72
  two: -529.54
  zero: -500.51
